In [8]:
import pandas as pd
bondora=pd.read_csv('Bondora_raw.csv')
print(bondora.head(2))

  ReportAsOfEOD                                LoanId  LoanNumber  \
0    2020-01-27  F0660C80-83F3-4A97-8DA0-9C250112D6EC         659   
1    2020-01-27  978BB85B-1C69-4D51-8447-9C240104A3A2         654   

           ListedOnUTC     BiddingStartedOn  BidsPortfolioManager  BidsApi  \
0  2009-06-11 16:40:39  2009-06-11 16:40:39                     0        0   
1  2009-06-10 15:48:57  2009-06-10 15:48:57                     0        0   

   BidsManual  UserName  NewCreditCustomer  ...  \
0    115.0410      KARU               True  ...   
1    140.6057  koort681              False  ...   

  PreviousEarlyRepaymentsCountBeforeLoan GracePeriodStart GracePeriodEnd  \
0                                    0.0              NaN            NaN   
1                                    0.0              NaN            NaN   

  NextPaymentDate NextPaymentNr NrOfScheduledPayments  ReScheduledOn  \
0             NaN           NaN                   NaN            NaN   
1             NaN           Na

/tmp/ipython-input-1564641376.py:2: DtypeWarning: Columns (85) have mixed types. Specify dtype option on import or set low_memory=False.
  bondora=pd.read_csv('Bondora_raw.csv')


A. Data Cleaning Phase
Start by cleaning the dataset to handle missing values, duplicates, and inconsistent formats.

Handle Missing Values

Identify columns with missing values using df.isnull().sum().

Drop columns with excessive missing values (>50% missing) if they don't add much value.

For numeric columns: Impute missing values using the median, mean, or a specific value.

For categorical columns: Impute missing values using the most frequent category or "Unknown."

In [9]:
#1.Identify columns with missing values
missing_values = bondora.isnull().sum()
print("Missing values in each column:\n", missing_values)

Missing values in each column:
 ReportAsOfEOD                             0
LoanId                                    0
LoanNumber                                0
ListedOnUTC                               0
BiddingStartedOn                          0
                                       ... 
NrOfScheduledPayments                  7293
ReScheduledOn                          9911
PrincipalDebtServicingCost                1
InterestAndPenaltyDebtServicingCost       1
ActiveLateLastPaymentCategory          6341
Length: 112, dtype: int64


In [12]:
# 2: Drop columns with more than 50% missing values
bondora=pd.read_csv('Bondora_raw.csv')
missing_values = bondora.isnull().sum() / len(bondora) * 100

cols_to_drop = missing_values[missing_values > 50].index.tolist()
cols_retained = missing_values[missing_values <= 50].index.tolist()

print("\nColumns with >50% missing values (Dropped):")
if cols_to_drop:
    for col in cols_to_drop:
        print(f"{col} - {missing_values[col]:.2f}% missing")
else:
    print("None")

print("\nColumns with ≤50% missing values (Retained):")
for col in cols_retained:
    print(f"{col} - {missing_values[col]:.2f}% missing")

bondora = bondora.drop(columns=cols_to_drop)

/tmp/ipython-input-2682029802.py:2: DtypeWarning: Columns (34,80,90) have mixed types. Specify dtype option on import or set low_memory=False.
  bondora=pd.read_csv('Bondora_raw.csv')



Columns with >50% missing values (Dropped):
EL_V0 - 99.23% missing
Rating_V0 - 99.23% missing
EL_V1 - 77.28% missing
Rating_V1 - 77.28% missing
Rating_V2 - 52.32% missing
CreditScoreEsMicroL - 65.49% missing
CreditScoreEsEquifaxRisk - 76.55% missing
CreditScoreFiAsiakasTietoRiskGrade - 82.75% missing
GracePeriodStart - 82.43% missing
GracePeriodEnd - 82.43% missing
NextPaymentDate - 85.81% missing
ReScheduledOn - 74.16% missing

Columns with ≤50% missing values (Retained):
ReportAsOfEOD - 0.00% missing
LoanId - 0.00% missing
LoanNumber - 0.00% missing
ListedOnUTC - 0.00% missing
BiddingStartedOn - 0.00% missing
BidsPortfolioManager - 0.00% missing
BidsApi - 0.00% missing
BidsManual - 0.00% missing
UserName - 0.00% missing
NewCreditCustomer - 0.00% missing
LoanApplicationStartedDate - 0.00% missing
LoanDate - 0.00% missing
ContractEndDate - 38.14% missing
FirstPaymentDate - 0.00% missing
MaturityDate_Original - 0.00% missing
MaturityDate_Last - 0.00% missing
ApplicationSignedHour - 0.0

In [13]:
# 3. Impute missing values for categorical columns
# Identify categorical columns
categorical_cols = bondora.select_dtypes(include=['object']).columns

print("\nCategorical columns:")

if categorical_cols.any():
    for col in categorical_cols:
        print(f"• {col}")

        # Impute missing values with the mode
        if bondora[col].isnull().sum() > 0:
            mode_val = bondora[col].mode()[0]
            bondora[col].fillna(mode_val, inplace=True)

    print("\nMissing values in categorical columns imputed using the most frequent category.")
else:
    print("\nNo categorical columns found for imputation.")

print(f"Dataset shape after categorical imputation: {bondora.shape}")



Categorical columns:
• ReportAsOfEOD
• LoanId
• ListedOnUTC
• BiddingStartedOn
• UserName
• LoanApplicationStartedDate
• LoanDate
• ContractEndDate
• FirstPaymentDate
• MaturityDate_Original
• MaturityDate_Last
• DateOfBirth
• Country
• County
• City
• NrOfDependants
• EmploymentDurationCurrentEmployer
• EmploymentPosition
• WorkExperience
• LastPaymentOn
• DebtOccuredOn
• DebtOccuredOnForSecondary
• DefaultDate
• StageActiveSince
• Rating
• Status
• ActiveLateCategory
• WorseLateCategory
• ActiveLateLastPaymentCategory

Missing values in categorical columns imputed using the most frequent category.
Dataset shape after categorical imputation: (32800, 100)


/tmp/ipython-input-488058420.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  bondora[col].fillna(mode_val, inplace=True)


In [14]:
# 4.Impute missing values for numeric columns
numeric_cols = bondora.select_dtypes(include=['int64', 'float64']).columns

print("\n--- Imputing Missing Values: Numeric Columns ---")
print("Numeric columns:")
for col in numeric_cols:
    print(f"• {col}")

# Impute missing values using the median
for col in numeric_cols:
    if bondora[col].isnull().sum() > 0:
        median_val = bondora[col].median()
        bondora[col].fillna(median_val, inplace=True)

print("\nMissing values in numeric columns imputed using the median.")
print(f"Dataset shape after numeric imputation: {bondora.shape}")



--- Imputing Missing Values: Numeric Columns ---
Numeric columns:
• LoanNumber
• BidsPortfolioManager
• BidsApi
• BidsManual
• ApplicationSignedHour
• ApplicationSignedWeekday
• VerificationType
• LanguageCode
• Age
• Gender
• AppliedAmount
• Amount
• Interest
• LoanDuration
• MonthlyPayment
• UseOfLoan
• Education
• MaritalStatus
• EmploymentStatus
• OccupationArea
• HomeOwnershipType
• IncomeFromPrincipalEmployer
• IncomeFromPension
• IncomeFromFamilyAllowance
• IncomeFromSocialWelfare
• IncomeFromLeavePay
• IncomeFromChildSupport
• IncomeOther
• IncomeTotal
• ExistingLiabilities
• LiabilitiesTotal
• RefinanceLiabilities
• DebtToIncome
• FreeCash
• MonthlyPaymentDay
• PlannedPrincipalTillDate
• PlannedInterestTillDate
• CurrentDebtDaysPrimary
• CurrentDebtDaysSecondary
• ExpectedLoss
• LossGivenDefault
• ExpectedReturn
• ProbabilityOfDefault
• PrincipalOverdueBySchedule
• PlannedPrincipalPostDefault
• PlannedInterestPostDefault
• EAD1
• EAD2
• PrincipalRecovery
• InterestRecovery
• 

/tmp/ipython-input-3195988845.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  bondora[col].fillna(median_val, inplace=True)


In [15]:
import pandas as pd

# Load dataset
bondora = pd.read_csv('Bondora_raw.csv')

# Drop columns with >50% missing values
bondora = bondora.drop(columns=bondora.columns[bondora.isnull().mean() > 0.5])

# Impute missing values
for col in bondora.columns:
    if bondora[col].dtype in ['int64', 'float64']:
        bondora[col].fillna(bondora[col].median(), inplace=True)
    else:
        mode_val = bondora[col].mode()[0] if not bondora[col].mode().empty else "Unknown"
        bondora[col].fillna(mode_val, inplace=True)

# Check missing values after imputation
print("\nMissing values after imputation:", bondora.isnull().sum().sum())


/tmp/ipython-input-25195621.py:4: DtypeWarning: Columns (34,80,85) have mixed types. Specify dtype option on import or set low_memory=False.
  bondora = pd.read_csv('Bondora_raw.csv')
/tmp/ipython-input-25195621.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  bondora[col].fillna(mode_val, inplace=True)
/tmp/ipython-input-25195621.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace metho


Missing values after imputation: 0


/tmp/ipython-input-25195621.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bondora[col].fillna(mode_val, inplace=True)


2.Remove Duplicates

Use df.duplicated() to check for duplicate rows.

Remove duplicates using df.drop_duplicates().

In [16]:
# Check for duplicate rows
duplicate_rows = bondora.duplicated()
print(f"Number of duplicate rows: {duplicate_rows.sum()}")

# Remove duplicate rows
bondora = bondora.drop_duplicates()


Number of duplicate rows: 0


3.Standardize Column Names

Rename columns to consistent lowercase/snake_case using df.columns.str.lower().str.replace(' ', '_').

In [17]:
bondora.columns = bondora.columns.str.lower().str.replace(' ', '_')
print(bondora.columns)

Index(['reportasofeod', 'loanid', 'loannumber', 'listedonutc',
       'biddingstartedon', 'bidsportfoliomanager', 'bidsapi', 'bidsmanual',
       'username', 'newcreditcustomer', 'loanapplicationstarteddate',
       'loandate', 'contractenddate', 'firstpaymentdate',
       'maturitydate_original', 'maturitydate_last', 'applicationsignedhour',
       'applicationsignedweekday', 'verificationtype', 'languagecode', 'age',
       'dateofbirth', 'gender', 'country', 'appliedamount', 'amount',
       'interest', 'loanduration', 'monthlypayment', 'county', 'city',
       'useofloan', 'education', 'maritalstatus', 'employmentstatus',
       'employmentdurationcurrentemployer', 'occupationarea',
       'homeownershiptype', 'incomefromprincipalemployer', 'incomefrompension',
       'incomefromfamilyallowance', 'incomefromsocialwelfare',
       'incomefromleavepay', 'incomefromchildsupport', 'incomeother',
       'incometotal', 'existingliabilities', 'liabilitiestotal',
       'refinanceliabiliti

4.Check Data Types

Ensure numeric columns are in the correct format using pd.to_numeric().

Convert date columns to datetime format using pd.to_datetime().

In [18]:
# Convert numeric columns to proper numeric types
numeric_cols = bondora.select_dtypes(include=['int64', 'float64', 'object']).columns
for col in numeric_cols:
    bondora[col] = pd.to_numeric(bondora[col], errors='ignore')  # only converts if possible

# Convert date columns to datetime
# Replace these with the actual date column names in your dataset
date_cols = ['date_column1', 'date_column2']  # example names
for col in date_cols:
    if col in bondora.columns:
        bondora[col] = pd.to_datetime(bondora[col], errors='coerce')  # invalid parsing becomes NaT

# Verify data types
print(bondora.dtypes)


reportasofeod                              object
loanid                                     object
loannumber                                  int64
listedonutc                                object
biddingstartedon                           object
                                           ...   
previousearlyrepaymentscountbeforeloan    float64
nextpaymentnr                             float64
nrofscheduledpayments                     float64
principaldebtservicingcost                float64
interestandpenaltydebtservicingcost       float64
Length: 85, dtype: object


/tmp/ipython-input-2240408823.py:4: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  bondora[col] = pd.to_numeric(bondora[col], errors='ignore')  # only converts if possible


2.Data Labeling for the Target Variable
The Status column needs to be converted into a binary variable (default or not default).

In [19]:
# Step 1: Examine unique values in 'Status'
print("Unique values in 'Status':")
print(bondora['status'].value_counts())

# Step 2: Define binary labels
# Map late or repaid statuses to 1 (default), current to 0 (not default)
status_mapping = {
    'Late': 1,
    'Repaid': 1,
    'Current': 0
}

# Create a new binary target column
bondora['target'] = bondora['status'].map(status_mapping)

# Step 3: Verify the distribution of the binary target
print("\nBinary target distribution (counts):")
print(bondora['target'].value_counts())

print("\nBinary target distribution (proportions):")
print(bondora['target'].value_counts(normalize=True))


Unique values in 'Status':
status
Late       32073
Repaid     24600
Current    18199
Name: count, dtype: int64

Binary target distribution (counts):
target
1    56673
0    18199
Name: count, dtype: int64

Binary target distribution (proportions):
target
1    0.756932
0    0.243068
Name: proportion, dtype: float64


Data Encoding
Convert categorical variables into numeric format.

1.Identify Categorical Variables


2.Encode Categorical Features:
  For binary categorical variables: Use label encoding.
  For multi-category variables: Use one-hot encoding.

In [20]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Identify categorical columns
categorical_cols = bondora.select_dtypes(include=['object']).columns
print("Categorical columns:", categorical_cols)

# Label encode a binary column
binary_column = 'target'  # replace with your actual binary column
if binary_column in bondora.columns:
    le = LabelEncoder()
    bondora[binary_column] = le.fit_transform(bondora[binary_column])
    print(f"Label encoded '{binary_column}'")
else:
    print(f"Column '{binary_column}' not found in bondora")

# One-hot encode a multi-category column
multi_category_column = 'Status'  # replace with your actual column
if multi_category_column in bondora.columns:
    bondora = pd.get_dummies(bondora, columns=[multi_category_column], drop_first=True)
    print(f"One-hot encoded '{multi_category_column}'")
else:
    print(f"Column '{multi_category_column}' not found in bondora")

# Verify the transformation
print(bondora.head(2))


Categorical columns: Index(['reportasofeod', 'loanid', 'listedonutc', 'biddingstartedon',
       'username', 'loanapplicationstarteddate', 'loandate', 'contractenddate',
       'firstpaymentdate', 'maturitydate_original', 'maturitydate_last',
       'dateofbirth', 'country', 'county', 'city',
       'employmentdurationcurrentemployer', 'lastpaymenton',
       'stageactivesince', 'rating', 'status', 'worselatecategory',
       'creditscoreesmicrol'],
      dtype='object')
Label encoded 'target'
Column 'Status' not found in bondora
  reportasofeod                                loanid  loannumber  \
0    2020-01-27  F0660C80-83F3-4A97-8DA0-9C250112D6EC         659   
1    2020-01-27  978BB85B-1C69-4D51-8447-9C240104A3A2         654   

           listedonutc     biddingstartedon  bidsportfoliomanager  bidsapi  \
0  2009-06-11 16:40:39  2009-06-11 16:40:39                     0        0   
1  2009-06-10 15:48:57  2009-06-10 15:48:57                     0        0   

   bidsmanual  userna

4. Data Cleaning for Outliers
Handle outliers in numeric features to improve model performance.

Detect Outliers

Use box plots or Z-scores.

Cap outliers to a certain percentile range:

In [21]:
# Identify numeric columns
numeric_cols = bondora.select_dtypes(include=['int64', 'float64']).columns
print("Numeric columns:", numeric_cols)

# Cap outliers at 1st and 99th percentiles
for col in numeric_cols:
    lower_bound = bondora[col].quantile(0.01)
    upper_bound = bondora[col].quantile(0.99)
    bondora[col] = bondora[col].clip(lower_bound, upper_bound)
    print(f"Capped outliers in '{col}' between {lower_bound} and {upper_bound}")

Numeric columns: Index(['loannumber', 'bidsportfoliomanager', 'bidsapi', 'bidsmanual',
       'applicationsignedhour', 'applicationsignedweekday', 'verificationtype',
       'languagecode', 'age', 'gender', 'appliedamount', 'amount', 'interest',
       'loanduration', 'monthlypayment', 'useofloan', 'education',
       'maritalstatus', 'employmentstatus', 'occupationarea',
       'homeownershiptype', 'incomefromprincipalemployer', 'incomefrompension',
       'incomefromfamilyallowance', 'incomefromsocialwelfare',
       'incomefromleavepay', 'incomefromchildsupport', 'incomeother',
       'incometotal', 'existingliabilities', 'liabilitiestotal',
       'refinanceliabilities', 'debttoincome', 'freecash', 'monthlypaymentday',
       'plannedprincipaltilldate', 'plannedinteresttilldate', 'expectedloss',
       'lossgivendefault', 'expectedreturn', 'probabilityofdefault',
       'principaloverduebyschedule', 'recoverystage', 'modelversion',
       'creditscoreeemini', 'principalpaymentsmade

In [ ]:
# Save Processed Data
bondora.to_csv('processed_bondora_data.csv', index=False)